# Pelajaran 09 - Pola Desain Metakognisi


## Pengaturan

Notebook ini menunjukkan pola desain Metacognition menggunakan Microsoft Agent Framework.

**Prasyarat:**
- Penyebaran Azure OpenAI dikonfigurasi melalui variabel lingkungan
- Azure CLI sudah diautentikasi (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Apa itu Metakognisi?

Metakognisi adalah **berpikir tentang berpikir**. Dalam konteks agen AI, ini berarti membangun agen yang dapat:

- **Merefleksikan diri** terhadap keluaran dan proses penalaran mereka sendiri
- **Mendeteksi kesalahan** dan pulih dengan anggun alih-alih gagal tanpa pemberitahuan
- **Mengevaluasi** apakah respons mereka lengkap dan membantu
- **Menyesuaikan** strategi mereka ketika pendekatan awal tidak berhasil (misalnya, beralih ke sistem cadangan)

Agen metakognitif tidak hanya menjawab pertanyaan — ia memantau kinerjanya sendiri dan menyesuaikan diri secara dinamis.


## Alat Utama dan Cadangan

Salah satu pola metakognisi yang umum adalah **strategi cadangan**. Agen mencoba alat utama terlebih dahulu; jika gagal (misalnya, error 404), agen mengenali kegagalan tersebut dan beralih secara transparan ke alat cadangan.

Hal ini mencerminkan sistem dunia nyata di mana layanan utama mungkin tidak tersedia dan agen harus mendiagnosis masalahnya sendiri sebelum memilih jalur alternatif.

Di bawah ini kami mendefinisikan dua alat pencarian penerbangan:
- **Utama** — mencakup Paris, Tokyo, dan Barcelona
- **Cadangan** — mencakup Berlin, Sydney, dan New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agen yang Merefleksikan Diri dengan Pemulihan Kesalahan

Agen di bawah ini diinstruksikan untuk mencoba sistem penerbangan utama terlebih dahulu, mengenali kegagalan, dan secara transparan beralih ke sistem cadangan. Setelah setiap respons, ia secara singkat mengevaluasi apakah ia telah sepenuhnya menjawab pertanyaan pengguna.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Pola Evaluasi Diri

Sisi lain dari metakognisi adalah **evaluasi diri**: agen terpisah (atau agen yang sama dalam lintasan kedua) meninjau sebuah respons untuk kelengkapan, ketepatan, dan kegunaan.

Di bawah ini kami membuat agen `ResponseEvaluator` yang memberi skor respons agen perjalanan pada tiga dimensi.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Ringkasan

Dalam pelajaran ini Anda mempelajari cara membangun **agen metakognitif** menggunakan Microsoft Agent Framework:

- **Refleksi diri**: Agen yang memantau penalaran mereka sendiri dan secara transparan mengomunikasikan apa yang terjadi.
- **Pemulihan kesalahan dengan fallback**: Pola alat utama + cadangan di mana agen mendeteksi kegagalan (mis. kesalahan 404) dan secara otomatis mencoba sumber alternatif.
- **Evaluasi diri**: Agen evaluator terpisah yang memberi skor respons berdasarkan kelengkapan, akurasi, dan kegunaan.

Pola-pola ini membuat agen lebih tangguh, transparan, dan dapat dipercaya — kualitas penting untuk penyebaran produksi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya mencapai ketepatan, harap diingat bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang otoritatif. Untuk informasi yang bersifat penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang salah yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
